In [1]:
import os
import json

os.makedirs("/root/.kaggle", exist_ok=True)

# Замени со твоите точни податоци
kaggle_creds = {
    "username": "svetlanamitkovska",
    "key": "KGAT_d9f4169d468a8f759c18fc904a8f17e1"
}

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_creds, f)

os.chmod("/root/.kaggle/kaggle.json", 0o600)

# Провери дали е правилно запишано
with open("/root/.kaggle/kaggle.json") as f:
    print("Содржина:", f.read())

Содржина: {"username": "svetlanamitkovska", "key": "KGAT_d9f4169d468a8f759c18fc904a8f17e1"}


In [2]:
# Инсталирање kaggle библиотека
!pip install kaggle -q

# Симнување датасетот директно во Colab
!kaggle datasets download -d renancostaalencar/compcars --path /content/compcars

print("Датасетот е симнат!")

Dataset URL: https://www.kaggle.com/datasets/renancostaalencar/compcars
License(s): unknown
100% 15.4G/15.4G [15:41<00:00, 17.6MB/s]

Датасетот е симнат!


In [3]:
import zipfile
import os

print("Распакувано")
with zipfile.ZipFile("/content/compcars/compcars.zip", "r") as zip_ref:
    zip_ref.extractall("/content/compcars/")

print("Готово!")
print(os.listdir("/content/compcars/"))

Распакувано
Готово!
['misc', 'part', 'image', 'compcars.zip', 'label', 'train_test_split']


In [4]:
# Importiranje na PyTorch
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [5]:
!pip install transformers timm -q
print("Готово!")

Готово!


In [6]:
# Vcituvanje train/test listi
BASE      = "/content/compcars"
IMAGE_DIR = f"{BASE}/image"
TRAIN_TXT = f"{BASE}/train_test_split/classification/train.txt"
TEST_TXT  = f"{BASE}/train_test_split/classification/test.txt"

with open(TRAIN_TXT) as f:
    train_lines = [l.strip() for l in f.readlines()]

with open(TEST_TXT) as f:
    test_lines = [l.strip() for l in f.readlines()]

print(f"Train слики: {len(train_lines)}")
print(f"Test слики:  {len(test_lines)}")
print(f"Пример: {train_lines[0]}")

Train слики: 16016
Test слики:  14939
Пример: 78/1/2010/439374a1456969.jpg


In [7]:
all_lines    = train_lines + test_lines
classes      = sorted(set("/".join(l.split("/")[:2]) for l in all_lines))
class_to_idx = {c: i for i, c in enumerate(classes)}

print(f"Вкупно класи: {len(classes)}")
print(f"Примери: {classes[:5]}")

Вкупно класи: 431
Примери: ['100/211', '100/212', '100/213', '100/249', '100/250']


In [8]:
#PyTorch Dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class CompCarsDataset(Dataset):
    def __init__(self, lines, image_dir, class_to_idx, transform=None):
        self.lines        = lines
        self.image_dir    = image_dir
        self.class_to_idx = class_to_idx
        self.transform    = transform

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, idx):
        line      = self.lines[idx]
        parts     = line.split("/")
        class_key = f"{parts[0]}/{parts[1]}"
        label     = self.class_to_idx[class_key]
        img_path  = os.path.join(self.image_dir, line)
        image     = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = CompCarsDataset(train_lines, IMAGE_DIR, class_to_idx, train_transforms)
test_dataset  = CompCarsDataset(test_lines,  IMAGE_DIR, class_to_idx, test_transforms)

print(f"Train: {len(train_dataset)} слики")
print(f"Test:  {len(test_dataset)} слики")

Train: 16016 слики
Test:  14939 слики


In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

print(f"Train батчеви: {len(train_loader)}")
print(f"Test батчеви:  {len(test_loader)}")

Train батчеви: 501
Test батчеви:  467


In [10]:
# Vcituvanje ViT model
from transformers import ViTForImageClassification

NUM_CLASSES = len(classes)

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)

print(f"Број на класи: {NUM_CLASSES}")
print(f"Број на параметри: {sum(p.numel() for p in model.parameters()):,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

[transformers] You passed `num_labels=431` which is incompatible to the `id2label` map of length `1000`.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([431, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([431])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Број на класи: 431
Број на параметри: 86,130,095


In [11]:
# Optimizer i Scheduler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

print(f"Модел на: {device}")
print("Подготвено!")

Модел на: cuda
Подготвено!


In [12]:
# Trening
from tqdm import tqdm

os.makedirs("checkpoints", exist_ok=True)

EPOCHS        = 10
best_accuracy = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct    = 0
    total      = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(pixel_values=images, labels=labels)
        loss    = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds       = outputs.logits.argmax(dim=-1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    train_acc  = correct / total * 100
    train_loss = total_loss / len(train_loader)

    model.eval()
    correct = 0
    total   = 0

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Eval]"):
            images  = images.to(device)
            labels  = labels.to(device)
            outputs = model(pixel_values=images)
            preds   = outputs.logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    test_acc = correct / total * 100
    scheduler.step()

    print(f"\nEpoch {epoch+1}: Loss={train_loss:.4f} | Train={train_acc:.2f}% | Test={test_acc:.2f}%")

    torch.save({
        "epoch":                epoch + 1,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_acc":            train_acc,
        "test_acc":             test_acc,
        "loss":                 train_loss,
    }, f"checkpoints/vit_compcars_epoch_{epoch+1}_acc{test_acc:.2f}.pt")
    print(f"  💾 Epoch {epoch+1} зачувана!")

    if test_acc > best_accuracy:
        best_accuracy = test_acc
        torch.save(model.state_dict(), "vit_compcars_best.pt")
        print(f"Нов најдобар модел! ({test_acc:.2f}%)")

Epoch 1/10 [Eval]: 100%|██████████| 467/467 [02:59<00:00,  2.61it/s]



Epoch 1: Loss=5.8265 | Train=2.03% | Test=5.55%
  💾 Epoch 1 зачувана!
Нов најдобар модел! (5.55%)


Epoch 2/10 [Eval]: 100%|██████████| 467/467 [02:57<00:00,  2.64it/s]



Epoch 2: Loss=5.0488 | Train=11.18% | Test=15.12%
  💾 Epoch 2 зачувана!
Нов најдобар модел! (15.12%)


Epoch 3/10 [Eval]: 100%|██████████| 467/467 [02:56<00:00,  2.64it/s]


...


Epoch 8: Loss=2.2698 | Train=71.81% | Test=52.69%
  💾 Epoch 8 зачувана!
Нов најдобар модел! (52.69%)


Epoch 9/10 [Eval]: 100%|██████████| 467/467 [02:56<00:00,  2.64it/s]



Epoch 9: Loss=2.1620 | Train=74.21% | Test=53.61%
  💾 Epoch 9 зачувана!
Нов најдобар модел! (53.61%)


Epoch 10/10 [Eval]: 100%|██████████| 467/467 [02:57<00:00,  2.63it/s]



Epoch 10: Loss=2.1171 | Train=75.49% | Test=53.92%
  💾 Epoch 10 зачувана!
Нов најдобар модел! (53.92%)


In [13]:
# Evaluacija
from sklearn.metrics import classification_report, top_k_accuracy_score
import numpy as np

model.eval()
all_preds  = []
all_labels = []
all_logits = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Евалуација"):
        images  = images.to(device)
        outputs = model(pixel_values=images)
        preds   = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_logits.extend(outputs.logits.cpu().numpy())

all_logits = np.array(all_logits)
top1 = np.mean(np.array(all_preds) == np.array(all_labels)) * 100
top5 = top_k_accuracy_score(all_labels, all_logits, k=5) * 100

print(f"Top-1 Accuracy: {top1:.2f}%")
print(f"Top-5 Accuracy: {top5:.2f}%")
print(classification_report(all_labels, all_preds, zero_division=0))

Евалуација: 100%|██████████| 467/467 [03:14<00:00,  2.40it/s]


Top-1 Accuracy: 53.92%
Top-5 Accuracy: 78.55%
              precision    recall  f1-score   support

           0       0.78      0.38      0.51        37
           1       0.00      0.00      0.00        12
           2       0.58      0.46      0.51        72
           3       1.00      0.07      0.13        14
           4       0.67      0.09      0.16        22
           5       0.39      0.62      0.48        78
           6       0.88      0.43      0.58        35
           7       0.95      0.75      0.84        24
           8       0.44      0.55      0.49        33
           9       0.52      0.62      0.57        42
          10       0.62      0.19      0.29        27
          11       1.00      0.20      0.33        20
          12       0.71      0.59      0.64        29
          13       1.00      0.31      0.47        13
          14       0.64      0.58      0.61        36
          15       0.62      0.61      0.62        62
          16       0.67      0.67  

In [ ]:
# Sporedba
import pandas as pd

results = {
    "Model":          ["EfficientNet-B0 v1", "EfficientNet-B0 v2", "ConvNeXt-Tiny", "ViT-B/16 CompCars"],
    "Dataset":        ["Stanford Cars",      "Stanford Cars",      "Stanford Cars",  "CompCars"],
    "Top-1 Accuracy": ["83.08%",             "81.64%",             "87.08%",         f"{top1:.2f}%"],
    "Top-5 Accuracy": ["95.20%",             "94.50%",             "96.81%",         f"{top5:.2f}%"],
    "Model Size":     ["17 MB",              "17 MB",              "112 MB",         "330 MB"],
}

df = pd.DataFrame(results)
print(df.to_string(index=False))